# 09a_fantrax_crosswalk

**Purpose:** Resolve Fantrax `scorer_id` to the player registries so
`fact_fantrax_adp` rows carry real foreign keys.

- Primary key: `gsis_id` via `dim_nfl_players` — the full nflverse registry already
  holds all veterans + signed rookies, so it covers ~100% of the board.
- Secondary: `player_key` via `dim_rookie_prospect` for current draft-class rookies.

**Matching:** exact cleaned-name -> disambiguate by position / active status /
recency -> fuzzy fallback (`token_sort_ratio`: auto >=90, review 70-89, unmatched <70).

**Outputs:**
- `data/dim_fantrax_crosswalk.parquet` — scorer_id -> gsis_id / player_key (+ method, score)
- `review_fantrax_crosswalk.csv` — rows needing manual review (only when any)
- back-fills `gsis_id` / `player_key` into `data/fact_fantrax_adp.parquet`


In [1]:
# ---- Setup & Config ---------------------------------------------------------
from dataclasses import dataclass
from datetime import date
from pathlib import Path
import re

import pandas as pd
from thefuzz import fuzz, process


@dataclass
class LeagueConfig:
    """Central config — paths + fuzzy thresholds for the Fantrax crosswalk."""
    draft_year: int = 2026
    data_dir: str = "data"
    fact_name: str = "fact_fantrax_adp"
    crosswalk_name: str = "dim_fantrax_crosswalk"
    nfl_players_name: str = "dim_nfl_players"
    rookie_prospect_name: str = "dim_rookie_prospect"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70

    def path(self, name: str) -> str:
        return f"{self.data_dir}/{name}.parquet"


CFG = LeagueConfig()


In [2]:
# ---- Helpers ----------------------------------------------------------------
_SUFFIX = re.compile(r"\b(jr|sr|ii|iii|iv|v)\b")


def clean_player_name(name: str) -> str:
    """Lowercase, drop periods/apostrophes and generational suffixes, collapse spaces."""
    if not isinstance(name, str):
        return ""
    n = name.lower().replace(".", "").replace("'", "").replace("\u2019", "")
    n = _SUFFIX.sub("", n)
    return re.sub(r"\s+", " ", n).strip()


def pos_tokens(raw: str) -> set:
    """Fantrax position string -> set of upper codes ('WR,DB' -> {WR, DB})."""
    return {p.strip().upper() for p in str(raw).split(",") if p.strip()}


In [3]:
# ---- Load -------------------------------------------------------------------
fact = pd.read_parquet(CFG.path(CFG.fact_name))
# one row per scorer_id (latest snapshot) — crosswalk is identity, not time series
fact_latest = (fact.sort_values(["season", "week"])
                    .drop_duplicates("scorer_id", keep="last"))

npl = pd.read_parquet(CFG.path(CFG.nfl_players_name)).copy()
rp = pd.read_parquet(CFG.path(CFG.rookie_prospect_name)).copy()

npl["name_clean"] = npl["display_name"].map(clean_player_name)
rp["name_clean"] = rp["player_name"].map(clean_player_name)

# rookie-prospect lookup: cleaned name -> player_key (first if dup)
rp_lookup = (rp.dropna(subset=["name_clean"])
               .drop_duplicates("name_clean")
               .set_index("name_clean")["player_key"].to_dict())

print(f"fantrax players: {len(fact_latest)} | dim_nfl_players: {len(npl)} | "
      f"dim_rookie_prospect: {len(rp)}")


fantrax players: 1653 | dim_nfl_players: 25036 | dim_rookie_prospect: 468


In [4]:
# ---- Match: scorer_id -> gsis_id (+ player_key) -----------------------------
# Index dim_nfl_players candidates by cleaned name for O(1) lookup.
npl_by_name = {n: g for n, g in npl.groupby("name_clean")}
npl_clean_names = list(npl_by_name.keys())


def disambiguate(cands: pd.DataFrame, fpos: set) -> pd.Series:
    """Pick the right candidate among same-name players: position, then active, then recency."""
    df = cands
    if fpos:
        m = (df["position"].str.upper().isin(fpos)
             | df["position_group"].str.upper().isin(fpos))
        if m.any():
            df = df[m]
    if (df["status"] == "ACT").any():
        df = df[df["status"] == "ACT"]
    if "entry_year" in df and df["entry_year"].notna().any():
        df = df.sort_values("entry_year", ascending=False)
    return df.iloc[0]


def match_one(row) -> dict:
    cn = clean_player_name(row["player_name"])
    fpos = pos_tokens(row["position_raw"])
    method, score, gsis = "unmatched", 0, None

    if cn in npl_by_name:
        cands = npl_by_name[cn]
        pick = cands.iloc[0] if len(cands) == 1 else disambiguate(cands, fpos)
        gsis = pick["gsis_id"]
        method = "exact" if len(cands) == 1 else "exact+disambig"
        score = 100
    else:
        best_name, best = process.extractOne(
            cn, npl_clean_names, scorer=fuzz.token_sort_ratio
        )
        score = best
        if best >= CFG.fuzzy_auto_threshold:
            pick = disambiguate(npl_by_name[best_name], fpos)
            gsis, method = pick["gsis_id"], "fuzzy"
        elif best >= CFG.fuzzy_review_threshold:
            method = "review"
        else:
            method = "unmatched"

    return {
        "scorer_id": row["scorer_id"],
        "player_name": row["player_name"],
        "position_raw": row["position_raw"],
        "nfl_team": row["nfl_team"],
        "is_rookie": row["is_rookie"],
        "gsis_id": gsis,
        "player_key": rp_lookup.get(cn),
        "match_method": method,
        "match_score": int(score),
    }


xwalk = pd.DataFrame([match_one(r) for _, r in fact_latest.iterrows()])
print(xwalk["match_method"].value_counts().to_string())


match_method
exact             1501
exact+disambig     115
review              29
fuzzy                8


In [5]:
# ---- Persist crosswalk + review staging -------------------------------------
xwalk["resolved_date"] = date.today().isoformat()
xwalk.to_parquet(CFG.path(CFG.crosswalk_name), index=False)

review = xwalk[xwalk["match_method"].isin(["review", "unmatched"])]
review_path = "data/review/review_fantrax_crosswalk.csv"
if len(review):
    out = review.copy()
    out["action"] = ""          # user fills: a gsis_id value, or "new"
    out.to_csv(review_path, index=False)
    print(f"[review] {len(review)} rows need manual review -> {review_path}")
else:
    print("[review] none — all rows auto-resolved")

print(f"[ok] wrote {len(xwalk)} crosswalk rows -> {CFG.path(CFG.crosswalk_name)}")


[review] 29 rows need manual review -> data/review/review_fantrax_crosswalk.csv
[ok] wrote 1653 crosswalk rows -> data/dim_fantrax_crosswalk.parquet


In [6]:
# ---- Back-fill FKs into fact_fantrax_adp ------------------------------------
fact = pd.read_parquet(CFG.path(CFG.fact_name))
key = xwalk.set_index("scorer_id")[["gsis_id", "player_key"]]
fact["gsis_id"] = fact["scorer_id"].map(key["gsis_id"])
fact["player_key"] = fact["scorer_id"].map(key["player_key"])
fact.to_parquet(CFG.path(CFG.fact_name), index=False)

print(f"[ok] back-filled fact_fantrax_adp — {fact['gsis_id'].notna().sum()}/"
      f"{len(fact)} rows have gsis_id "
      f"({fact['gsis_id'].notna().mean() * 100:.1f}%)")


[ok] back-filled fact_fantrax_adp — 1624/1653 rows have gsis_id (98.2%)


In [7]:
# ---- Validation report ------------------------------------------------------
print("=== crosswalk coverage ===")
print(f"total scorer_ids:       {len(xwalk)}")
print(f"gsis_id resolved:       {xwalk['gsis_id'].notna().sum()}")
print(f"player_key (rookies):   {xwalk['player_key'].notna().sum()}")
print(f"needs review/unmatched: {xwalk['match_method'].isin(['review', 'unmatched']).sum()}")
print()
print("by method:")
print(xwalk["match_method"].value_counts().to_string())
print()
# sanity: any gsis_id claimed by >1 scorer_id?
dups = xwalk[xwalk["gsis_id"].notna()].groupby("gsis_id").size()
dups = dups[dups > 1]
if len(dups):
    print(f"[warn] {len(dups)} gsis_id mapped by >1 scorer_id:")
    print(xwalk[xwalk["gsis_id"].isin(dups.index)]
          [["player_name", "gsis_id", "match_method"]].to_string())
else:
    print("[ok] gsis_id mapping is 1:1")


=== crosswalk coverage ===
total scorer_ids:       1653
gsis_id resolved:       1624
player_key (rookies):   224
needs review/unmatched: 29

by method:
match_method
exact             1501
exact+disambig     115
review              29
fuzzy                8

[warn] 1 gsis_id mapped by >1 scorer_id:
       player_name     gsis_id    match_method
797   Jaylon Jones  00-0038407  exact+disambig
1072  Jaylon Jones  00-0038407  exact+disambig
